<a href="https://colab.research.google.com/github/SaieshNeeli/Attendence-through-CV/blob/master/transfer_learning_for_cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GOU check


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import os
import time

from torchvision import datasets, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Device:", device)

PyTorch version: 2.14.1+cu130
Torchvision version: 0.29.1+cu130
Device: cuda


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

!nvidia-smi

CUDA available: True
CUDA version: 13.0
GPU count: 1
Tue Nov 10 17:44:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N

In [4]:
import urllib.request
import zipfile

url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

urllib.request.urlretrieve(url, "hymenoptera_data.zip")

with zipfile.ZipFile("hymenoptera_data.zip", "r") as zip_ref:
    zip_ref.extractall("data")

data_dir = "data/hymenoptera_data"

print("Dataset downloaded successfully!")
print("Training classes:", os.listdir(data_dir + "/train"))
print("Validation classes:", os.listdir(data_dir + "/val"))

Dataset downloaded successfully!
Training classes: ['bees', 'ants']
Validation classes: ['bees', 'ants']


In [5]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ]),

    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )
    ])
}

In [6]:
image_datasets = {
    x: datasets.ImageFolder(
        os.path.join(data_dir, x),
        data_transforms[x]
    )
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(
        image_datasets[x],
        batch_size=4,
        shuffle=True if x == 'train' else False,
        num_workers=2
    )
    for x in ['train', 'val']
}

dataset_sizes = {
    x: len(image_datasets[x])
    for x in ['train', 'val']
}

class_names = image_datasets['train'].classes

print("Classes:", class_names)
print("Training images:", dataset_sizes['train'])
print("Validation images:", dataset_sizes['val'])

Classes: ['ants', 'bees']
Training images: 244
Validation images: 153


In [7]:
inputs, labels = next(iter(dataloaders['train']))

print("Input shape:", inputs.shape)
print("Labels shape:", labels.shape)
print("Labels:", labels)

Input shape: torch.Size([4, 3, 224, 224])
Labels shape: torch.Size([4])
Labels: tensor([0, 0, 1, 0])


In [8]:
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model = model.to(device)

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [9]:
weights=models.ResNet18_Weights.DEFAULT

In [10]:
print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


What does this mean?
- in_features=512: ResNet18 generates 512 features for each image.
- out_features=1000: It predicts 1,000 ImageNet categories.
- fc: Fully connected classification layer.

In [11]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


In [12]:
for name, param in model.named_parameters():
    print(name, param.requires_grad)

conv1.weight True
bn1.weight True
bn1.bias True
layer1.0.conv1.weight True
layer1.0.bn1.weight True
layer1.0.bn1.bias True
layer1.0.conv2.weight True
layer1.0.bn2.weight True
layer1.0.bn2.bias True
layer1.1.conv1.weight True
layer1.1.bn1.weight True
layer1.1.bn1.bias True
layer1.1.conv2.weight True
layer1.1.bn2.weight True
layer1.1.bn2.bias True
layer2.0.conv1.weight True
layer2.0.bn1.weight True
layer2.0.bn1.bias True
layer2.0.conv2.weight True
layer2.0.bn2.weight True
layer2.0.bn2.bias True
layer2.0.downsample.0.weight True
layer2.0.downsample.1.weight True
layer2.0.downsample.1.bias True
layer2.1.conv1.weight True
layer2.1.bn1.weight True
layer2.1.bn1.bias True
layer2.1.conv2.weight True
layer2.1.bn2.weight True
layer2.1.bn2.bias True
layer3.0.conv1.weight True
layer3.0.bn1.weight True
layer3.0.bn1.bias True
layer3.0.conv2.weight True
layer3.0.bn2.weight True
layer3.0.bn2.bias True
layer3.0.downsample.0.weight True
layer3.0.downsample.1.weight True
layer3.0.downsample.1.bias True
la

Method 1 — Fine-tuning

In [16]:
model1 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model1.fc = nn.Linear(model1.fc.in_features, 2)

model1 = model1.to(device)

Method 2 — Fixed feature extractor.
Only the newly created final layer is trainable.

In [17]:
model2 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

for param in model2.parameters():
    param.requires_grad = False

model2.fc = nn.Linear(model2.fc.in_features, 2)

model2 = model2.to(device)

In [18]:
print("MODEL 1")
print("Trainable:", sum(p.numel() for p in model1.parameters() if p.requires_grad))
print("Total:", sum(p.numel() for p in model1.parameters()))

print("\nMODEL 2")
print("Trainable:", sum(p.numel() for p in model2.parameters() if p.requires_grad))
print("Total:", sum(p.numel() for p in model2.parameters()))

MODEL 1
Trainable: 11177538
Total: 11177538

MODEL 2
Trainable: 1026
Total: 11177538


In [19]:
criterion = nn.CrossEntropyLoss()

optimizer1 = optim.SGD(
    model1.parameters(),
    lr=0.001,
    momentum=0.9
)

optimizer2 = optim.SGD(
    model2.fc.parameters(),
    lr=0.001,
    momentum=0.9
)

Understand these three concepts
1. Loss function

CrossEntropyLoss() measures how far the model's predictions are from the correct class.

2. Learning rate

lr=0.001 controls the size of each weight update.

3. Momentum

momentum=0.9 helps SGD maintain a direction of progress using previous gradients.

7.2 Understand each operation
Code	Meaning
model.train()	Enable training mode
optimizer.zero_grad()	Clear previous gradients
model(inputs)	Forward propagation
criterion(outputs, labels)	Calculate loss
loss.backward()	Calculate gradients
optimizer.step()	Update weights
model.eval()	Enable evaluation mode
torch.no_grad()	Disable gradient calculation

In [20]:
def train_model(model, optimizer, num_epochs=10):

    since = time.time()

    for epoch in range(num_epochs):

        print(f"\nEpoch {epoch+1}/{num_epochs}")

        for phase in ['train', 'val']:

            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:

                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):

                    outputs = model(inputs)

                    loss = criterion(outputs, labels)

                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)

                running_corrects += torch.sum(
                    preds == labels.data
                )

            epoch_loss = running_loss / dataset_sizes[phase]

            epoch_acc = running_corrects.item() / dataset_sizes[phase]

            print(
                f"{phase} Loss: {epoch_loss:.4f} "
                f"Acc: {epoch_acc:.4f}"
            )

    time_elapsed = time.time() - since

    print(f"\nTraining completed in {time_elapsed // 60:.0f}m "
          f"{time_elapsed % 60:.0f}s")

    return model

In [22]:
model1 = train_model(
    model1,
    optimizer1,
    num_epochs=10
)


Epoch 1/10
train Loss: 0.6030 Acc: 0.7049
val Loss: 0.5231 Acc: 0.7778

Epoch 2/10
train Loss: 0.4409 Acc: 0.7992
val Loss: 0.2135 Acc: 0.9412

Epoch 3/10
train Loss: 0.7030 Acc: 0.7377
val Loss: 0.6143 Acc: 0.8170

Epoch 4/10
train Loss: 0.7708 Acc: 0.7090
val Loss: 1.7378 Acc: 0.5229

Epoch 5/10
train Loss: 0.5665 Acc: 0.7910
val Loss: 0.4386 Acc: 0.8824

Epoch 6/10
train Loss: 0.6018 Acc: 0.7992
val Loss: 0.2785 Acc: 0.8758

Epoch 7/10
train Loss: 0.7850 Acc: 0.7377
val Loss: 0.7951 Acc: 0.8170

Epoch 8/10
train Loss: 0.6940 Acc: 0.7541
val Loss: 0.3366 Acc: 0.8758

Epoch 9/10
train Loss: 0.5842 Acc: 0.8115
val Loss: 0.2966 Acc: 0.8693

Epoch 10/10
train Loss: 0.7037 Acc: 0.7787
val Loss: 0.3228 Acc: 0.8562

Training completed in 0m 32s


In [23]:
model2 = train_model(
    model2,
    optimizer2,
    num_epochs=10
)


Epoch 1/10
train Loss: 1.0772 Acc: 0.5451
val Loss: 0.2394 Acc: 0.9281

Epoch 2/10
train Loss: 0.4856 Acc: 0.7623
val Loss: 0.2115 Acc: 0.9346

Epoch 3/10
train Loss: 0.4469 Acc: 0.7664
val Loss: 0.1930 Acc: 0.9412

Epoch 4/10
train Loss: 0.6287 Acc: 0.7541
val Loss: 0.1941 Acc: 0.9542

Epoch 5/10
train Loss: 0.3649 Acc: 0.8074
val Loss: 0.2099 Acc: 0.9346

Epoch 6/10
train Loss: 0.4002 Acc: 0.8115
val Loss: 0.2384 Acc: 0.9216

Epoch 7/10
train Loss: 0.4373 Acc: 0.7910
val Loss: 0.2019 Acc: 0.9412

Epoch 8/10
train Loss: 0.4777 Acc: 0.7951
val Loss: 0.2048 Acc: 0.9412

Epoch 9/10
train Loss: 0.5106 Acc: 0.8033
val Loss: 0.2122 Acc: 0.9346

Epoch 10/10
train Loss: 0.4640 Acc: 0.8033
val Loss: 0.2200 Acc: 0.9412

Training completed in 0m 27s
